## 0. colab 설치 하기

1. Google Drive로 이동
2. +새로만들기 (New) 클릭
3. 더보기 - 연결할 앱 더보기 클릭
4. Colaboratory 검색
5. "추가" 버튼 클릭하기

## 1. colab의 하드웨어 성능

### OS 환경

In [7]:
import platform
print(platform.platform())

Windows-10-10.0.26200-SP0


###  CPU 사양

In [6]:
try:
    import psutil
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "psutil"], check=False)
    import psutil
import platform
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
freq = psutil.cpu_freq()
if freq:
    print(f"Max freq: {freq.max/1000:.2f} GHz, Current: {freq.current/1000:.2f} GHz")
print(f"Processor: {platform.processor() or platform.machine()}")

Physical cores: 16
Logical cores: 24
Max freq: 2.20 GHz, Current: 2.20 GHz
Processor: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel


### 메모리 사양

In [8]:
try:
    import psutil
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "psutil"], check=False)
    import psutil
m = psutil.virtual_memory()
print(f"Total: {m.total/1e9:.2f} GB, Available: {m.available/1e9:.2f} GB, Used: {m.used/1e9:.2f} GB")

Total: 16.87 GB, Available: 5.22 GB, Used: 11.65 GB


### python 버젼

In [4]:
!python --version

Python 3.10.11


In [ ]:
# 시스템 사양/가속 가능 여부 요약
import platform, subprocess, sys

# psutil 준비
try:
    import psutil
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'psutil'], check=False)
    import psutil

info = {}
info['os'] = platform.platform()
info['cpu_physical'] = psutil.cpu_count(logical=False)
info['cpu_logical'] = psutil.cpu_count(logical=True)
freq = psutil.cpu_freq()
info['cpu_freq_GHz'] = round((freq.current/1000.0), 2) if freq else None
vm = psutil.virtual_memory()
info['ram_GB'] = round(vm.total/1e9, 2)

# GPU 확인
cuda_available = False
gpu_name = None
gpu_mem = None

# 1) PyTorch가 있다면 가장 정확
try:
    import torch  # type: ignore
    cuda_available = bool(torch.cuda.is_available())
    if cuda_available:
        idx = 0
        gpu_name = torch.cuda.get_device_name(idx)
        gpu_mem = round(torch.cuda.get_device_properties(idx).total_memory/1e9, 2)
except Exception:
    pass

# 2) TensorFlow로 확인 시도
if not cuda_available:
    try:
        import tensorflow as tf  # type: ignore
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            cuda_available = True
            gpu_name = getattr(gpus[0], 'name', 'GPU')
    except Exception:
        pass

# 3) OS별 커맨드로 확인 (Windows wmic / nvidia-smi)
if not cuda_available:
    try:
        if platform.system() == 'Windows':
            out = subprocess.check_output(['wmic', 'path', 'win32_VideoController', 'get', 'name'], encoding='utf-8', errors='ignore')
            lines = [l.strip() for l in out.splitlines() if l.strip() and 'Name' not in l]
            if lines:
                gpu_name = lines[0]
        else:
            out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], encoding='utf-8', errors='ignore')
            parts = out.split(',')
            if parts:
                gpu_name = parts[0].strip()
                if len(parts) > 1:
                    gpu_mem = parts[1].strip()
    except Exception:
        pass

print('=== System Summary ===')
print(f"OS: {info['os']}")
print(f"CPU: {info['cpu_physical']} physical / {info['cpu_logical']} logical, ~{info['cpu_freq_GHz']} GHz")
print(f"RAM: {info['ram_GB']} GB")
print(f"CUDA GPU Available: {cuda_available}")
print(f"GPU: {gpu_name} | VRAM: {gpu_mem} GB" if gpu_name else 'GPU: Not detected')

print('\n=== Guidance ===')
if cuda_available:
    print('- 딥러닝/대규모 행렬 연산: GPU 사용 시 5~50배 가속 가능')
    print('- VRAM 한도 내에서 배치/모델 크기 조절 필요')
else:
    print('- 로컬 GPU 미감지: 딥러닝 학습은 CPU-only로 느릴 수 있음')
    print('- 필요 시 Colab에서 GPU/TPU 런타임 사용 권장')
print('- 판다스/일반 데이터 전처리는 CPU/메모리/디스크 속도에 좌우됩니다')



## 1-1. GPU 사용하기

수정 -> 노트 설정 -> 하드웨어 가속기 -> GPU 설정

In [9]:
# GPU 사용 가능 여부와 GPU 정보 출력
try:
    import tensorflow as tf
    if tf.config.list_physical_devices('GPU'):
        gpu_devices = tf.config.list_physical_devices('GPU')
        print(f"GPU 사용 가능: {len(gpu_devices)}개")
        for i, gpu in enumerate(gpu_devices):
            print(f"{i+1}번째 GPU 정보:", gpu)
    else:
        print("GPU를 사용할 수 없습니다.")
except ImportError:
    try:
        import torch
        if torch.cuda.is_available():
            print(f"GPU 사용 가능: {torch.cuda.get_device_name(0)}")
        else:
            print("GPU를 사용할 수 없습니다.")
    except ImportError:
        try:
            from tensorflow.python.client import device_lib
            local_device_protos = device_lib.list_local_devices()
            gpu_list = [x for x in local_device_protos if x.device_type == 'GPU']
            if gpu_list:
                print("GPU 사용 가능:", gpu_list)
            else:
                print("GPU를 사용할 수 없습니다.")
        except Exception as e:
            print("GPU 정보 확인을 위한 패키지가 없습니다.")


GPU 사용 가능: NVIDIA GeForce RTX 4060 Laptop GPU


In [11]:
pip install tensorflow


  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
   ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
    --------------------------------------- 7.6/331.7 MB 39.0 MB/s eta 0:00:09
   - -------------------------------------- 14.4/331.7 MB 36.2 MB/s eta 0:00:09
   -- ------------------------------------- 22.8/331.7 MB 38.0 MB/s eta 0:00:09
   --- ------------------------------------ 31.5/331.7 MB 39.1 MB/s eta 0:00:08
   ---- ----------------------------------- 40.9/331.7 MB 40.6 MB/s eta 0:00:08
   ----- ---------------------------------- 48.0/331.7 MB 39.7 MB/s eta 0:00:08
   ------ --------------------------------- 56.1/331.7 MB 39.3 MB/s eta 0:00:08
   ------- -------------------------------- 65.0/331.7 MB 39.8 MB/s eta 0:00:07
   -------- ------------------------------- 73.4/331.7 MB 40.0 MB/s eta 0:00:07
   ---------- ----------------------------- 83.1/331.7 MB 40.8 MB/s eta 0:00:07
   ----------- ---------------------------- 93.3/331.7 MB 41.7 MB/s e


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# '수정 -> 노트 설정 -> 하드웨어 가속기 -> GPU 설정' 이 설명대로 진행했다면, 아래 코드를 통해 GPU가 제대로 할당되었는지 확인하세요.

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU가 정상적으로 할당되었습니다. 사용 가능한 GPU 개수: {len(gpus)}")
    for idx, gpu in enumerate(gpus):
        print(f"{idx+1}번째 GPU 정보: {gpu}")
else:
    print("GPU가 할당되지 않았습니다. '노트 설정'에서 하드웨어 가속기를 GPU로 변경했는지 다시 확인해보세요.")


GPU가 할당되지 않았습니다. '노트 설정'에서 하드웨어 가속기를 GPU로 변경했는지 다시 확인해보세요.


## 2. 기본 메뉴 설명

1. 목차
2. 파일 탐색기
3. 할당 여부 확인 하기 (RAM/디스크)
4. 댓글 (전체 파일에 대하여 / 셀에 대하여)

## 3. 공유받기 / 공유하기


### 공유받은 colab을 읽기 전용 -> 쓰기로 출력하기

1. 파일 - 드라이브에 사본 저장을 클릭하여 "내 드라이브"에 추가 합니다
2. 파일 - Locate in Drive (드라이브 위치)에서 확인을 클릭 (파일의 저장 위치 확인 가능)

### 내가 작성한 파일을 다른 사람과 공유하기

우측 상단에 "공유" 버튼을 클릭하여, 공유가능하며, 링크로 전달 할 수 있습니다.

**[주의]**
공유 받은 사람이 쓰기 권한이 있으면, 파일이 맘대로 수정될 수 있으니, 유의하세요

## 4. 셀의 종류 (코드/텍스트)

코드 셀: 코드 실행을 위한 셀

텍스트 셀: 주석을 달거나, 설명 문구등을 달기 위한 텍스트 전용 셀

### 4-1. 주석과 텍스트 셀의 차이

코드셀 설명입니다.

In [5]:
print('안녕하세요')

안녕하세요


In [0]:
# 이렇게 주석을 달 수 있습니다.
print('안녕하세요')

이것은 텍스트 셀입니다

In [0]:
# 주석입니다.
print('안녕하세요')

# 주석입니다.
print('안녕하세요')

### 4-2. 셀의 삽입 방법

상단의 "+코드", "+텍스트"를 눌러서 삽입합니다

**[셀 추가 관련 단축키]**

1. 코드 셀 위에 삽입 (Ctrl/Cmd + M A)
2. 코드 셀 아래에 삽입 (Ctrl/Cmd + M B)


텍스트

### 4-3. 셀의 삭제 방법

우측의 **휴지통 아이콘을 클릭**하여 삭제합니다.

**[셀 삭제 관련 단축키]**

셀 지우기 (Ctrl/Cmd + M D)

### 4-4 셀의 타입 변경

**[셀 타입 변경 단축키]**
1. 코드 셀로 변경 (Ctrl/Cmd + M Y)
2. 마크다운 셀로 변경 (Ctrl/Cmd + M M)

print('안녕하세요')

## 5. 실행과 출력

* **셀**: 실행의 단위
* **실행**: 재생 버튼을 클릭하여, 실행 (해당 셀 실행)
* **실행 순서**: *인터프리터 방식* 으로 실행하며, 순차적으로 메모리에 기억합니다.


**[실행 관련 단축키]**

1. 해당 셀을 실행하고 커서는 해당 셀  (Ctrl + Enter)
2. 해당 셀을 실행하고 커서는 다음 셀  (Shift + Enter)
3. 해당 셀을 실행하고 커서는 다음에 셀 삽입  (Alt + Enter)

In [0]:
print('안녕하세요1')

## 6. 혹시 단축키를 모른다면?

Ctrl + M H (키보드 단축키 보기)

## 7. 마크다운 문법의 이해

Markdown(마크다운)이란 Markup(마크업)언어의 일종으로, **HTML경험이 없는 사람도 누구나 쉽게** 헤더,글 머리 기호,이미지 삽입, 링크, 글자 모양 등 다양한 서식을 쉽게 추가하는 방식의 문서 편집 문법이다.

**[실습예제]**
## colab 이란?
Google Colaboratory

구글에서 **교육과 과학 연구를 목적**으로 개발한 도구로, 2017년에 무료로 공개하였습니다.

### 장점
1. 무료
2. 구글 드라이브와 연동 가능
3. GPU를 사용할 수 있음
4. 깃헙(Github)에 있는 노트북 코드를 바로 돌려볼 수 있음
5. 손 쉽게 타인과 공유할 수 있음


### 단점
1. 90분동안 아무런 인터액션이 없거나 (idle timeout), 12시간의 ***세션 타임아웃***이 존재함
2. GPU는 할당을 못받을 수도 있음 (혹은 빼앗기는 경우도 있음)
3. 조금 느림



# 헤더

구글에서 교육과 **과학 연구**를 목적으로 개발한 도구로, 2017년에 무료로 공개하였습니다.

1. 순서1
2. 순서1
3. 순서1


* 순서없다
* 순서없다
* 순서없다

## 8. 파일 업로드 하기

### 방법 1: 직접 업로드하기

In [0]:
from google.colab import files
myfile = files.upload()

In [0]:
import io
import pandas as pd

In [0]:
data = pd.read_csv(io.BytesIO(myfile['house_price.csv']))

In [0]:
data.head()

### 방법 2: 드라이브에서 가져오기

In [0]:
from google.colab import drive
drive.mount('/content/drive')

In [0]:
filename = '/content/drive/My Drive/01_fastcampus/ch01-google-colaboratory/샘플데이터/house_price.csv'

In [0]:
data = pd.read_csv(filename)

In [0]:
data.head()

## 9. 그 밖의 기능 - 이미지, Youtube 동영상 로딩하기

In [0]:
from IPython.display import Image

In [0]:
Image('https://i.pinimg.com/736x/0b/2f/8a/0b2f8a51314ab1ebe0505aee843a33b1.jpg')

In [0]:
from IPython.display import YouTubeVideo

In [0]:
YouTubeVideo('0FswLjHeMUk', 600, 380)

In [0]:
from IPython.display import HTML

In [0]:
HTML('https://www.fastcampus.co.kr/')

In [13]:
# TensorFlow 설치 위치/버전 확인
import sys, os, subprocess
print('Python:', sys.executable)
print('Env PATH head:', os.environ.get('PATH','').split(os.pathsep)[:3])

try:
    import tensorflow as tf
    tf_path = tf.__file__
    print('TensorFlow version:', tf.__version__)
    print('TensorFlow module file:', tf_path)
    print('TensorFlow package root:', os.path.dirname(os.path.dirname(tf_path)))
except Exception as e:
    print('TensorFlow import failed:', e)

# pip show 로도 위치 확인
try:
    out = subprocess.check_output([sys.executable, '-m', 'pip', 'show', 'tensorflow'], encoding='utf-8', errors='ignore')
    print('\n[pip show tensorflow]\n', out)
except Exception as e:
    print('pip show tensorflow failed:', e)



Python: c:\Users\HA\AppData\Local\Programs\Python\Python310\python.exe
Env PATH head: ['c:\\Users\\HA\\AppData\\Local\\Programs\\Python\\Python310', 'c:\\Users\\HA\\AppData\\Roaming\\Python\\Python310\\Scripts', 'C:\\WINDOWS\\system32']
TensorFlow version: 2.20.0
TensorFlow module file: c:\Users\HA\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\__init__.py
TensorFlow package root: c:\Users\HA\AppData\Local\Programs\Python\Python310\lib\site-packages

[pip show tensorflow]
 Name: tensorflow
Version: 2.20.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: c:\users\ha\appdata\local\programs\python\python310\lib\site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termc

In [14]:
# 용량/자원 현황 자동 확인 (TensorFlow 설치 용량, GPU VRAM, RAM/디스크)
import os, pathlib, subprocess, sys

# TensorFlow 설치 폴더 용량
try:
    import tensorflow as tf
    tf_root = pathlib.Path(tf.__file__).parent.parent
    tf_size = 0
    for p in tf_root.rglob('*'):
        try:
            if p.is_file():
                tf_size += p.stat().st_size
        except Exception:
            pass
    print('[TensorFlow] path:', tf_root)
    print('[TensorFlow] size:', f'{tf_size/1024/1024:.1f} MB')
except Exception as e:
    print('TensorFlow not importable:', e)

# GPU VRAM
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print('[GPU]', props.name, '| VRAM:', f'{props.total_memory/1e9:.2f} GB')
    else:
        raise RuntimeError('CUDA not available')
except Exception:
    try:
        out = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], encoding='utf-8', errors='ignore')
        print('[GPU via nvidia-smi]', out.strip())
    except Exception as e:
        print('GPU info unavailable:', e)

# RAM/디스크
try:
    import psutil, shutil
except ImportError:
    import pip
    pip.main(['install','psutil'])
    import psutil, shutil

ram = psutil.virtual_memory()
print('[RAM] total:', f'{ram.total/1e9:.2f} GB', '| available:', f'{ram.available/1e9:.2f} GB')

total, used, free = shutil.disk_usage(os.getcwd())
print('[Disk] cwd:', os.getcwd())
print('[Disk] total:', f'{total/1e9:.2f} GB', '| free:', f'{free/1e9:.2f} GB')



[TensorFlow] path: c:\Users\HA\AppData\Local\Programs\Python\Python310\lib\site-packages
[TensorFlow] size: 6731.9 MB
[GPU] NVIDIA GeForce RTX 4060 Laptop GPU | VRAM: 8.59 GB
[RAM] total: 16.87 GB | available: 6.07 GB
[Disk] cwd: c:\Study\Python\[전체강의자료]직장인을-위한-파이썬-데이터분석\01. Part1.[파이썬 필수 스킬] 데이터 분석을 위한 준비\ch01.google_colaboratory\해설코드
[Disk] total: 511.19 GB | free: 360.72 GB
